In [1]:
# ============================================================
# CELL 1 — KẾT NỐI VÀ KIỂM TRA DATASET TAOBAO CLEAN
# ============================================================

from pathlib import Path
import pyarrow.dataset as ds
import json

# 1. Đường dẫn Dataset trên Kaggle
DATASET_DIR = Path(
    "/kaggle/input/datasets/huy2008/taobao-clean"
)

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Không tìm thấy Dataset: {DATASET_DIR}"
    )

# 2. Chỉ tìm các file Parquet
parquet_files = sorted(
    DATASET_DIR.rglob("part_*.parquet")
)

print("=" * 60)
print("KIỂM TRA DATASET TAOBAO CLEAN")
print("=" * 60)

print("Số file Parquet:", len(parquet_files))

if len(parquet_files) != 201:
    raise ValueError(
        f"Dự kiến 201 file, thực tế {len(parquet_files)}"
    )

# 3. Chỉ đọc danh sách file Parquet
# Không đọc cleaning_report.json hoặc cleaning_summary.csv
dataset = ds.dataset(
    [str(file) for file in parquet_files],
    format="parquet"
)

# 4. Kiểm tra số tương tác
total = dataset.count_rows()

print("Tổng tương tác:", f"{total:,}")

if total != 100_095_231:
    raise ValueError(
        f"Số tương tác không khớp: {total:,}"
    )

# 5. Kiểm tra cấu trúc dữ liệu
print("\nSCHEMA:")
print(dataset.schema)

# 6. Đọc báo cáo làm sạch
report_files = list(
    DATASET_DIR.rglob("cleaning_report.json")
)

if report_files:
    with open(report_files[0], "r", encoding="utf-8") as f:
        cleaning_report = json.load(f)

    print("\nCLEANING REPORT:")
    print(
        json.dumps(
            cleaning_report,
            indent=2,
            ensure_ascii=False
        )
    )

# 7. Hoàn tất
print("\n" + "=" * 60)
print("KẾT NỐI DATASET THÀNH CÔNG")
print("=" * 60)

ModuleNotFoundError: No module named 'pyarrow'

In [ ]:
# ============================================================
# CELL 2 — KIỂM TRA CẤU TRÚC DỮ LIỆU SAU LÀM SẠCH
# ============================================================

import pandas as pd
import pyarrow.parquet as pq
from collections import Counter

# Yêu cầu Cell 1 đã chạy thành công
if "parquet_files" not in globals():
    raise RuntimeError(
        "Cần chạy Cell 1 trước để tạo parquet_files."
    )

if not parquet_files:
    raise ValueError("Danh sách file Parquet đang rỗng.")

print("=" * 65)
print("CELL 2 — KIỂM TRA DỮ LIỆU TAOBAO CLEAN")
print("=" * 65)

# 1. Đọc metadata của tất cả file Parquet
schemas = []
total_rows = 0

for path in parquet_files:
    pf = pq.ParquetFile(str(path))
    total_rows += pf.metadata.num_rows
    schemas.append(pf.schema_arrow)

# 2. Kiểm tra schema nhất quán
reference_schema = schemas[0]

for idx, schema in enumerate(schemas[1:], start=2):
    if not schema.equals(reference_schema):
        raise ValueError(
            f"Schema không đồng nhất tại file thứ {idx}: "
            f"{parquet_files[idx - 1].name}"
        )

print("\n1. THÔNG TIN DATASET")
print(f"Số file Parquet: {len(parquet_files):,}")
print(f"Tổng số tương tác: {total_rows:,}")

assert total_rows == 100_095_231, (
    f"Số dòng không khớp: {total_rows:,}"
)

# 3. Kiểm tra tên cột và kiểu dữ liệu
print("\n2. CẤU TRÚC CỘT")
for field in reference_schema:
    print(f"  {field.name}: {field.type}")

# 4. Đọc một batch nhỏ để xem dữ liệu mẫu
# Không tải toàn bộ Dataset vào RAM
first_file = pq.ParquetFile(str(parquet_files[0]))
batch_iterator = first_file.iter_batches(batch_size=10)

try:
    first_batch = next(batch_iterator)
except StopIteration:
    raise ValueError("File Parquet đầu tiên không có dữ liệu.")

sample_df = first_batch.to_pandas()

print("\n3. DỮ LIỆU MẪU")
print(sample_df.to_string(index=False))

print("\n4. KIỂU DỮ LIỆU PANDAS")
print(sample_df.dtypes.to_string())

# 5. Kiểm tra các cột quan trọng
columns = set(reference_schema.names)

column_candidates = {
    "user": ["user_id", "User_ID", "userid"],
    "item": ["item_id", "Item_ID", "itemid"],
    "behavior": [
        "behavior_type", "behavior", "Behavior"
    ],
    "timestamp": [
        "timestamp", "Timestamp", "time"
    ]
}

detected_columns = {}

print("\n5. NHẬN DIỆN CÁC CỘT ĐỒ THỊ")

for role, candidates in column_candidates.items():
    found = next(
        (name for name in candidates if name in columns),
        None
    )
    detected_columns[role] = found
    print(f"  {role}: {found or 'CHƯA XÁC ĐỊNH'}")

# 6. Kiểm tra timestamp trên mẫu
time_col = detected_columns["timestamp"]

if time_col is not None:
    print("\n6. KIỂM TRA TIMESTAMP MẪU")

    values = sample_df[time_col]

    if pd.api.types.is_numeric_dtype(values):
        time_sample = pd.to_datetime(
            values,
            unit="s",
            utc=True,
            errors="coerce"
        )
    else:
        time_sample = pd.to_datetime(
            values,
            utc=True,
            errors="coerce"
        )

    print(
        time_sample.dt.tz_convert(
            "Asia/Shanghai"
        ).to_string(index=False)
    )
else:
    print("\n6. Chưa xác định được cột timestamp.")

# 7. Kiểm tra điều kiện chuẩn bị xây dựng đồ thị
missing_roles = [
    role for role, name in detected_columns.items()
    if name is None
]

print("\n" + "=" * 65)

if missing_roles:
    print("CẦN XÁC NHẬN TÊN CỘT:", missing_roles)
else:
    print("CELL 2 COMPLETED SUCCESSFULLY")
    print("DỮ LIỆU SẴN SÀNG CHO BƯỚC ÁNH XẠ NODE")

print("=" * 65)

In [ ]:
# ============================================================
# CELL 3 — TEMPORAL TRAIN / VALIDATION / TEST SPLIT
# Taobao UserBehavior — 100% cleaned dataset
# Timezone: Asia/Shanghai (UTC+8)
# ============================================================

from pathlib import Path
from collections import Counter
from datetime import datetime, timezone, timedelta
import json
import time

import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

# 1. Kiểm tra dữ liệu đầu vào
if "parquet_files" not in globals():
    raise RuntimeError("Hãy chạy Cell 1 trước.")

if len(parquet_files) != 201:
    raise ValueError(
        f"Dự kiến 201 file, nhận {len(parquet_files)}"
    )

required_columns = {
    "user_id", "item_id", "category_id",
    "behavior", "timestamp", "behavior_id"
}

schema = pq.read_schema(str(parquet_files[0]))
assert required_columns.issubset(schema.names)
assert pa.types.is_integer(schema.field("timestamp").type)

# 2. Mốc thời gian — tính bằng datetime để tránh sai epoch
TZ = timezone(timedelta(hours=8))

def to_epoch(year, month, day):
    return int(
        datetime(
            year, month, day,
            tzinfo=TZ
        ).timestamp()
    )

TRAIN_START = to_epoch(2017, 11, 25)
VAL_START = to_epoch(2017, 12, 1)
TEST_START = to_epoch(2017, 12, 2)
DATA_END = to_epoch(2017, 12, 4)

SPLITS = {
    "train": (TRAIN_START, VAL_START),
    "validation": (VAL_START, TEST_START),
    "test": (TEST_START, DATA_END),
}

# 3. Output
OUTPUT_DIR = Path("/kaggle/working/taobao_temporal_split")

if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError(
        f"Output đã tồn tại: {OUTPUT_DIR}. "
        "Không tự động ghi đè dữ liệu cũ."
    )

for name in SPLITS:
    (OUTPUT_DIR / name).mkdir(parents=True, exist_ok=True)

# 4. Thống kê
stats = {
    name: {
        "rows": 0,
        "files": 0,
        "min_timestamp": None,
        "max_timestamp": None,
        "behavior_counts": Counter()
    }
    for name in SPLITS
}

total_input = 0
start_time = time.time()

# 5. Xử lý theo batch, không tải toàn bộ vào RAM
for file_idx, file_path in enumerate(parquet_files):

    parquet_file = pq.ParquetFile(str(file_path))

    if not parquet_file.schema_arrow.equals(schema):
        raise ValueError(f"Schema không khớp: {file_path}")

    for batch_idx, batch in enumerate(
        parquet_file.iter_batches(
            batch_size=250_000,
            use_threads=False
        )
    ):
        table = pa.Table.from_batches([batch])
        total_input += table.num_rows
        timestamps = table["timestamp"]

        for name, (lower, upper) in SPLITS.items():

            mask = pc.and_(
                pc.greater_equal(timestamps, lower),
                pc.less(timestamps, upper)
            )

            subset = table.filter(mask)

            if subset.num_rows == 0:
                continue

            ts_range = pc.min_max(
                subset["timestamp"]
            ).as_py()

            min_ts = ts_range["min"]
            max_ts = ts_range["max"]

            assert lower <= min_ts <= max_ts < upper

            output_file = (
                OUTPUT_DIR / name /
                f"part_{file_idx:04d}_{batch_idx:04d}.parquet"
            )

            pq.write_table(
                subset,
                str(output_file),
                compression="zstd"
            )

            s = stats[name]
            s["rows"] += subset.num_rows
            s["files"] += 1

            s["min_timestamp"] = (
                min_ts if s["min_timestamp"] is None
                else min(s["min_timestamp"], min_ts)
            )

            s["max_timestamp"] = (
                max_ts if s["max_timestamp"] is None
                else max(s["max_timestamp"], max_ts)
            )

            for entry in pc.value_counts(
                subset["behavior"]
            ).to_pylist():
                s["behavior_counts"][entry["values"]] += (
                    entry["counts"]
                )

    if (file_idx + 1) % 20 == 0:
        print(
            f"Processed {file_idx + 1}/{len(parquet_files)} files",
            flush=True
        )

# 6. Kiểm tra toàn vẹn
total_output = sum(s["rows"] for s in stats.values())

assert total_input == 100_095_231, (
    f"Số dòng đầu vào không khớp: {total_input:,}"
)

assert total_output == total_input, (
    f"Mất dữ liệu: {total_input - total_output:,} dòng"
)

for name, s in stats.items():
    assert s["rows"] > 0, f"{name} không có dữ liệu"
    assert sum(s["behavior_counts"].values()) == s["rows"]

# 7. Lưu báo cáo
report = {
    "total_input": total_input,
    "total_output": total_output,
    "timezone": "Asia/Shanghai",
    "split_boundaries": {
        name: {"start": lower, "end_exclusive": upper}
        for name, (lower, upper) in SPLITS.items()
    },
    "splits": {
        name: {
            "rows": s["rows"],
            "files": s["files"],
            "min_timestamp": s["min_timestamp"],
            "max_timestamp": s["max_timestamp"],
            "behavior_counts": dict(s["behavior_counts"])
        }
        for name, s in stats.items()
    },
    "processing_minutes": round(
        (time.time() - start_time) / 60, 2
    )
}

with open(
    OUTPUT_DIR / "temporal_split_report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

# 8. Hiển thị kết quả
print("\n" + "=" * 65)
print("CELL 3 — TEMPORAL SPLIT RESULTS")
print("=" * 65)

for name, s in stats.items():
    print(f"\n{name.upper()}")
    print(f"  Interactions : {s['rows']:,}")
    print(f"  Parquet files: {s['files']}")
    print(f"  Min timestamp: {s['min_timestamp']}")
    print(f"  Max timestamp: {s['max_timestamp']}")

    for behavior, count in sorted(
        s["behavior_counts"].items()
    ):
        print(f"  {behavior}: {count:,}")

print(f"\nTOTAL INPUT : {total_input:,}")
print(f"TOTAL OUTPUT: {total_output:,}")
print(f"OUTPUT: {OUTPUT_DIR}")
print("\nCELL 3 COMPLETED SUCCESSFULLY")